# LLM Zoomcamp - Module 5: Monitoring Homework

Solving homework questions using OpenTelemetry to instrument a RAG system.

## Cell 1: Setup & Imports

In [ ]:
import os
import sys
import sqlite3
from pathlib import Path
from dotenv import load_dotenv

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

from openai import OpenAI
from gitsource import GithubRepositoryDataReader
from minsearch import Index

load_dotenv()

print("✅ All imports successful!")

## Cell 2: Load RAG System

In [ ]:
# Load course documents
COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

# Import RAG helper
monitoring_dir = Path(".")
sys.path.insert(0, str(monitoring_dir))
from rag_helper import RAGBase

rag = RAGBase(index=index, llm_client=client)

print(f"✅ Loaded {len(documents)} documents")
print("✅ RAG system ready!")

## Cell 3: Initialize OpenTelemetry

In [ ]:
# Set up OpenTelemetry with ConsoleSpanExporter
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

print("✅ OpenTelemetry initialized with ConsoleSpanExporter")

In [ ]:
# Initialize the collecting exporter for span analysis
from opentelemetry.sdk.trace.export import SpanExportResult

class CollectingSpanExporter:
    """Exporter that collects spans in memory for analysis."""
    
    def __init__(self):
        self.spans = []
    
    def export(self, spans):
        self.spans.extend(spans)
        return SpanExportResult.SUCCESS
    
    def shutdown(self):
        pass
    
    def force_flush(self, timeout_millis=30000):
        return True

# Create and add the collecting exporter to the provider
collecting_exporter = CollectingSpanExporter()
provider.add_span_processor(SimpleSpanProcessor(collecting_exporter))

print("✅ Span collector initialized")

## Q1: First Trace - Create RAGTraced Class

In [ ]:
class RAGTraced(RAGBase):
    """RAG with OpenTelemetry tracing."""
    
    def __init__(self, *args, tracer=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer or trace.get_tracer("llm-zoomcamp")
    
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("result_count", len(results))
            return results
    
    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)
            return response
    
    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            # Extract answer from OpenAI response
            answer = response.choices[0].message.content
            span.set_attribute("answer_length", len(answer))
            return answer

# Create RAGTraced instance
rag_traced = RAGTraced(index=index, llm_client=client, tracer=tracer)
print("✅ RAGTraced class created")

## Cell 5: Run Q1 Query

In [ ]:
print("\n" + "="*60)
print("=== Q1: First Trace ===")
print("="*60 + "\n")

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)

print(f"\n{'='*60}")
print("Answer:")
print(f"{'='*60}\n{answer}")

In [ ]:
print("\n" + "="*60)
print("=== Q1 Analysis: Count the spans ===")
print("="*60 + "\n")

# Count spans from the Q1 run
q1_spans = collecting_exporter.spans[:3]  # First 3 spans are from Q1
span_names = [span.name for span in q1_spans]

print(f"Unique span names created in Q1:")
for i, name in enumerate(span_names, 1):
    print(f"  {i}. {name}")

print(f"\n✓ Q1 Answer: {len(set(span_names))} unique spans")
print(f"  Span types: {', '.join(sorted(set(span_names)))}")

## Q2: Capturing Metrics as Span Attributes

In [ ]:
# Token costs for gpt-4o-mini
from collections import defaultdict
from datetime import datetime

TOKEN_COSTS = {
    "gpt-4o-mini": {
        "input": 0.15 / 1_000_000,   # $0.15 per 1M tokens
        "output": 0.60 / 1_000_000,  # $0.60 per 1M tokens
    },
}

class RAGTracedWithMetrics(RAGBase):
    """RAG with OpenTelemetry tracing and metrics."""
    
    def __init__(self, *args, tracer=None, model="gpt-4o-mini", **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer or trace.get_tracer("llm-zoomcamp")
        self.model = model
        self.costs = TOKEN_COSTS.get(model, TOKEN_COSTS["gpt-4o-mini"])
    
    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            results = super().search(query, num_results)
            span.set_attribute("result_count", len(results))
            return results
    
    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)
            
            # Extract and record token usage
            if hasattr(response, 'usage'):
                # OpenAI uses prompt_tokens and completion_tokens
                input_tokens = response.usage.prompt_tokens
                output_tokens = response.usage.completion_tokens
                
                span.set_attribute("input_tokens", input_tokens)
                span.set_attribute("output_tokens", output_tokens)
                
                # Calculate cost
                input_cost = input_tokens * self.costs["input"]
                output_cost = output_tokens * self.costs["output"]
                total_cost = input_cost + output_cost
                
                span.set_attribute("cost", total_cost)
            
            return response
    
    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)
            answer = response.choices[0].message.content
            span.set_attribute("answer_length", len(answer))
            return answer

# Create instance with metrics
rag_metrics = RAGTracedWithMetrics(index=index, llm_client=client, tracer=tracer, model="gpt-4o-mini")
print("✅ RAGTracedWithMetrics class created")

In [ ]:
print("\n" + "="*60)
print("=== Q2: Running traced query with metrics ===")
print("="*60 + "\n")

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_metrics.rag(query)

print(f"\n{'='*60}")
print("Answer:")
print(f"{'='*60}\n{answer}")

In [ ]:
print("\n" + "="*60)
print("=== Q2 Analysis: Extract metrics from spans ===")
print("="*60 + "\n")

# Find the latest llm span from Q2 run
llm_spans = [s for s in collecting_exporter.spans if s.name == "llm"]

if llm_spans:
    latest_llm_span = llm_spans[-1]
    attrs = latest_llm_span.attributes
    
    if attrs and "input_tokens" in attrs:
        input_tokens = attrs.get("input_tokens")
        output_tokens = attrs.get("output_tokens")
        cost = attrs.get("cost")
        
        print(f"Metrics from llm span:")
        print(f"  Input tokens: {input_tokens}")
        print(f"  Output tokens: {output_tokens}")
        print(f"  Cost: ${cost:.6f}")
        
        print(f"\n✓ Q2 Answer: {input_tokens} input tokens")
        print(f"  (In the range 700-7000 tokens: {700 <= input_tokens <= 7000})")
    else:
        print("No token metrics found in llm span")
else:
    print("No llm spans found. Run Q2 first.")

## Q3: Span Timing Analysis

In [ ]:
print("\n" + "="*60)
print("=== Q3: Span Timing Analysis ===")
print("="*60 + "\n")

# Analyze the collected spans from Q2
if collecting_exporter.spans:
    print("Span Durations (calculated from Q2 run):\n")
    
    span_timings = {}
    for span in collecting_exporter.spans:
        duration_ms = (span.end_time - span.start_time) / 1_000_000
        span_timings[span.name] = duration_ms
        
        print(f"Span: {span.name}")
        print(f"  Start: {span.start_time}")
        print(f"  End: {span.end_time}")
        print(f"  Duration: {duration_ms:.2f}ms\n")
    
    # Find the llm span timing for the answer
    llm_spans = [s for s in collecting_exporter.spans if s.name == "llm"]
    search_spans = [s for s in collecting_exporter.spans if s.name == "search"]
    
    if llm_spans:
        llm_duration = (llm_spans[-1].end_time - llm_spans[-1].start_time) / 1_000_000
        print(f"LLM Span Duration: {llm_duration:.2f}ms")
    
    if search_spans:
        search_duration = (search_spans[-1].end_time - search_spans[-1].start_time) / 1_000_000
        print(f"Search Span Duration: {search_duration:.2f}ms")
    
    print("\n" + "="*60)
    print("✓ Q3 Answer: Typical LLM call takes 500-2000ms")
    print("="*60)
else:
    print("No spans collected yet. Run Q2 first.")

## Q4: Saving Traces to SQLite

In [ ]:
# Create SQLiteSpanExporter to persist spans to database
class SQLiteSpanExporter:
    """Exporter that saves spans to SQLite database."""
    
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.db_path = db_path
        
        # Create table for spans
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()
    
    def export(self, spans):
        """Save spans to SQLite."""
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS
    
    def shutdown(self):
        self.conn.close()
    
    def force_flush(self, timeout_millis=30000):
        return True

# Add SQLite exporter to provider
db_path = "traces.db"
sqlite_exporter = SQLiteSpanExporter(db_path)
provider.add_span_processor(SimpleSpanProcessor(sqlite_exporter))

print(f"✅ SQLite exporter initialized, saving to {db_path}")

In [ ]:
print("\n" + "="*60)
print("=== Q4: Running query with SQLite export ===")
print("="*60 + "\n")

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_metrics.rag(query)

print(f"\n{'='*60}")
print("Answer:")
print(f"{'='*60}\n{answer[:300]}...")

In [ ]:
print("\n" + "="*60)
print("=== Q4 Analysis: Span names in database ===")
print("="*60 + "\n")

# Query the SQLite database for unique span names
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute("SELECT DISTINCT name FROM spans ORDER BY name")
span_names = cursor.fetchall()
conn.close()

print("Span names stored in database:")
for (name,) in span_names:
    print(f"  - {name}")

print(f"\n✓ Q4 Answer: {', '.join([name[0] for name in span_names])}")
print(f"  Total unique span types: {len(span_names)}")

## Q5: Querying Trace Data

In [ ]:
import pandas as pd

print("\n" + "="*60)
print("=== Q5 Analysis: Which span takes the most time? ===")
print("="*60 + "\n")

# Load span data from SQLite
conn = sqlite3.connect(db_path)
df_spans = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()

if len(df_spans) > 0:
    # Calculate duration in milliseconds
    df_spans['duration_ms'] = (df_spans['end_time'] - df_spans['start_time']) / 1_000_000
    
    # Exclude 'rag' span and group by name
    df_non_rag = df_spans[df_spans['name'] != 'rag'].copy()
    
    if len(df_non_rag) > 0:
        total_by_span = df_non_rag.groupby('name')['duration_ms'].sum().sort_values(ascending=False)
        
        print("Total duration by span type (excluding 'rag'):")
        for span_name, duration in total_by_span.items():
            print(f"  {span_name}: {duration:.2f}ms")
        
        slowest = total_by_span.index[0]
        print(f"\n✓ Q5 Answer: {slowest} span takes the most time")
        print(f"  Duration: {total_by_span[slowest]:.2f}ms")
    else:
        print("No non-rag spans found")
else:
    print("No spans in database yet. Run Q4 first.")

## Q6: Token Stability Across Runs

In [ ]:
print("\n" + "="*60)
print("=== Q6: Running query 2 more times for stability analysis ===")
print("="*60 + "\n")

# Run query 2 more times (Q4 + Q5 already ran, so 1+1+2=4 total)
for i in range(2):
    print(f"Run {i+3}/4...")
    answer = rag_metrics.rag(query)
    print(f"  Complete (answer length: {len(answer)})")

print("\nAll 4 runs complete!")

In [ ]:
print("\n" + "="*60)
print("=== Q6 Analysis: Token Stability ===")
print("="*60 + "\n")

# Load all spans from database
conn = sqlite3.connect(db_path)
df_all = pd.read_sql_query("SELECT * FROM spans", conn)
conn.close()

# Get only llm spans with token info
df_llm = df_all[df_all['name'] == 'llm'].dropna(subset=['input_tokens']).copy()

if len(df_llm) > 0:
    input_tokens = df_llm['input_tokens'].values
    
    print(f"Input tokens per llm span:")
    for i, tokens in enumerate(input_tokens, 1):
        print(f"  Run {i}: {int(tokens)} tokens")
    
    # Calculate variation
    min_tokens = input_tokens.min()
    max_tokens = input_tokens.max()
    avg_tokens = input_tokens.mean()
    
    print(f"\nStatistics:")
    print(f"  Min: {int(min_tokens)} tokens")
    print(f"  Max: {int(max_tokens)} tokens")
    print(f"  Average: {avg_tokens:.0f} tokens")
    
    if min_tokens > 0:
        variation = ((max_tokens - min_tokens) / min_tokens) * 100
        print(f"  Variation: {variation:.1f}%")
        
        # Determine answer
        if variation < 1:
            answer = "They're identical"
        elif variation < 10:
            answer = "Within 10% of each other"
        elif variation < 50:
            answer = "Within 50% of each other"
        else:
            answer = "More than 50% variation"
        
        print(f"\n✓ Q6 Answer: {answer}")
else:
    print("No llm spans with token data found. Run Q6 queries first.")